<a href="https://colab.research.google.com/github/fidlarsyn/Introduction-Machine-Learning-with-python/blob/main/BAB_5_Model_Evaluation_and__Improvement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Teknik Cross-Validation (Validasi Silang)**
Validasi silang adalah metode statistik untuk mengevaluasi kinerja generalisasi yang lebih stabil daripada pembagian data train-test tunggal.


# Cross-Validation Dasar
Menggunakan cross_val_score. Secara default pada versi terbaru, scikit-learn menggunakan 5-fold (sebelumnya 3-fold pada versi 0.18).

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

# Memuat dataset Iris (Multikelas)
iris = load_iris()
logreg = LogisticRegression()

# Menjalankan cross-validation
# Catatan: cv=5 adalah standar industri saat ini untuk mengurangi varians evaluasi
scores = cross_val_score(logreg, iris.data, iris.target, cv=5)

print("Skor Cross-validation per fold: {}".format(scores))
print("Rata-rata skor cross-validation: {:.3f}".format(scores.mean()))

# K-Fold Cross-Validation dengan Shuffling
Gunakan objek KFold secara eksplisit jika data Anda terurut (seperti dataset Iris) untuk menghindari bias kelas pada tiap fold.

In [ ]:
from sklearn.model_selection import KFold

# Menggunakan shuffle=True untuk mengacak data sebelum dibagi menjadi fold
# random_state dipastikan tetap (fixed) untuk hasil yang dapat direproduksi (reproducibility)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(logreg, iris.data, iris.target, cv=kfold)
print("Skor Cross-validation (KFold dengan Shuffle): {}".format(scores))

# Stratified K-Fold
Dalam tugas klasifikasi, sangat disarankan menggunakan stratified agar proporsi kelas di setiap fold mencerminkan proporsi dataset asli. Ini adalah perilaku default cross_val_score untuk klasifikasi.

In [ ]:
from sklearn.model_selection import StratifiedKFold

# StratifiedKFold memastikan tiap fold memiliki representasi kelas yang seimbang
strat_kfold = StratifiedKFold(n_splits=3)
scores = cross_val_score(logreg, iris.data, iris.target, cv=strat_kfold)
print("Skor Stratified CV: {}".format(scores))

# Shuffle-Split & GroupKFold
ShuffleSplit memungkinkan kontrol iterasi yang fleksibel, sementara GroupKFold sangat krusial jika data Anda memiliki ketergantungan kelompok (misal: beberapa foto dari pasien yang sama).

In [ ]:
import numpy as np
from sklearn.model_selection import ShuffleSplit, GroupKFold

# ShuffleSplit: Mengambil 50% data training dan 20% data test sebanyak 10 kali secara acak
shuffle_split = ShuffleSplit(test_size=.2, train_size=.5, n_splits=10, random_state=0)
scores = cross_val_score(logreg, iris.data, iris.target, cv=shuffle_split)
print("Skor Shuffle-Split (10 iterasi): \n{}".format(scores))

# GroupKFold: Menghindari kebocoran data (data leakage) antar grup
# Misal: Kita memiliki 150 sampel yang dibagi ke dalam 3 grup besar (misal 3 lokasi berbeda)
groups = np.repeat(np.arange(3), 50)
group_kfold = GroupKFold(n_splits=3)
scores = cross_val_score(logreg, iris.data, iris.target, groups=groups, cv=group_kfold)
print("Skor GroupKFold: {}".format(scores))

# **Pencarian Parameter (Grid Search)**
Mencari kombinasi parameter terbaik (seperti C dan gamma pada SVM) sangat penting untuk performa model.

# Simple Grid Search (Manual)
Pendekatan ini menunjukkan bahaya mengevaluasi parameter langsung pada test set.

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

# Pembagian data awal
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, random_state=0)
best_score = 0

# Loop melalui kombinasi parameter
for gamma in [0.001, 0.01, 0.1, 1, 10]:
    for C in [0.001, 0.01, 0.1, 1, 10]:
        svm = SVC(gamma=gamma, C=C)
        svm.fit(X_train, y_train)
        # Bahaya: Jika kita menggunakan X_test di sini, kita 'membocorkan' informasi test set ke model
        score = svm.score(X_test, y_test)
        if score > best_score:
            best_score = score
            best_parameters = {'C': C, 'gamma': gamma}

print("Skor terbaik (Manual): {:.2f}".format(best_score))
print("Parameter terbaik: {}".format(best_parameters))

# Implementasi GridSearchCV (Otomatis & Kuat)
Golden Rule: Gunakan validasi silang di dalam pencarian parameter pada data latih, dan simpan test set hanya untuk evaluasi akhir.

In [ ]:
from sklearn.model_selection import GridSearchCV
import pandas as pd

# Mendefinisikan grid parameter dalam bentuk dictionary
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10],
              'gamma': [0.001, 0.01, 0.1, 1, 10]}

# Inisialisasi GridSearchCV dengan CV=5
grid_search = GridSearchCV(SVC(), param_grid, cv=5)

# Fit dilakukan hanya pada data training
grid_search.fit(X_train, y_train)

# Analisis Hasil
print("Skor test set akhir: {:.2f}".format(grid_search.score(X_test, y_test)))
print("Parameter terbaik: {}".format(grid_search.best_params_))
print("Skor validasi terbaik: {:.2f}".format(grid_search.best_score_))

# Menampilkan hasil dalam tabel Markdown
results_df = pd.DataFrame(grid_search.cv_results_)
print("\nRingkasan Hasil Grid Search (5 baris teratas):")
print(results_df[['param_C', 'param_gamma', 'mean_test_score', 'rank_test_score']].head())

# **Metrik Evaluasi dan Skor**
Akurasi bisa menipu pada dataset yang tidak seimbang (imbalanced). Kita akan menggunakan dataset Breast Cancer (biner) untuk visualisasi metrik yang lebih tepat.

# Confusion Matrix dan Visualisasi
Mengukur jenis kesalahan (FP, FN, TP, TN) secara visual.

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.datasets import load_breast_cancer
import matplotlib.pyplot as plt

# Muat data kanker untuk klasifikasi biner
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(cancer.data, cancer.target, random_state=0)

logreg = LogisticRegression(max_iter=10000).fit(X_train, y_train)
pred = logreg.predict(X_test)
confusion = confusion_matrix(y_test, pred)

# Visualisasi Confusion Matrix menggunakan Matplotlib
plt.figure(figsize=(6, 4))
plt.imshow(confusion, interpolation='nearest', cmap=plt.cm.Blues)
plt.title("Confusion Matrix (Breast Cancer)")
plt.colorbar()
tick_marks = np.arange(len(cancer.target_names))
plt.xticks(tick_marks, cancer.target_names)
plt.yticks(tick_marks, cancer.target_names)
plt.ylabel('Label Sebenarnya')
plt.xlabel('Label Prediksi')

# Menambahkan teks angka di dalam kotak
for i in range(confusion.shape[0]):
    for j in range(confusion.shape[1]):
        plt.text(j, i, str(confusion[i, j]), horizontalalignment="center")

plt.show()

# Precision, Recall, dan F1-Score
classification_report memberikan ringkasan statistik yang krusial untuk memahami trade-off antara presisi dan recall.

In [ ]:
from sklearn.metrics import classification_report

# Menampilkan laporan lengkap
# Precision: Kemampuan model tidak melabeli sampel negatif sebagai positif
# Recall: Kemampuan model menemukan semua sampel positif
print(classification_report(y_test, pred, target_names=cancer.target_names))

# ROC Curve dan AUC Score
Evaluasi performa klasifikasi biner pada berbagai ambang batas (threshold).

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Mendapatkan nilai kepastian (decision function) dari SVC
svc = SVC(gamma=0.01).fit(X_train, y_train)
dec_func = svc.decision_function(X_test)

# Menghitung False Positive Rate dan True Positive Rate
fpr, tpr, thresholds = roc_curve(y_test, dec_func)

# Visualisasi ROC Curve
plt.plot(fpr, tpr, label="ROC Curve")
plt.xlabel("FPR (False Positive Rate)")
plt.ylabel("TPR (Recall)")
plt.title("ROC Curve - SVC")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray') # Garis diagonal baseline
plt.legend(loc=4)
plt.show()

# Menghitung Area Under the Curve (AUC)
# AUC mendekati 1 berarti model memiliki kemampuan pemisahan kelas yang sangat baik
auc_score = roc_auc_score(y_test, dec_func)
print("Skor AUC: {:.3f}".format(auc_score))

# Metrik Multikelas (Iris Dataset)
Menggunakan rata-rata untuk data lebih dari dua kelas.

In [ ]:
from sklearn.metrics import f1_score

# Prediksi pada dataset Iris (3 kelas)
pred_iris = LogisticRegression(max_iter=1000).fit(X_train, y_train).predict(X_test)

# Macro: menghitung skor per kelas lalu dirata-rata (cocok jika tiap kelas sama pentingnya)
print("F1-score Macro: {:.3f}".format(f1_score(y_test, pred, average='macro')))
# Micro: menghitung total TP, FN, dan FP secara global (cocok jika ada imbalance)
print("F1-score Micro: {:.3f}".format(f1_score(y_test, pred, average='micro')))

# **Integrasi Metrik dalam Seleksi Model**
Mengoptimalkan model berdasarkan metrik tertentu (bukan hanya akurasi) menggunakan parameter scoring.


# Parameter Scoring pada GridSearchCV
Sangat berguna ketika biaya kesalahan False Negative berbeda dengan False Positive.

In [ ]:
# Mencari parameter terbaik menggunakan metrik AUC (untuk klasifikasi biner)
param_grid_svc = {'C': [0.1, 1, 10], 'gamma': [0.001, 0.01, 0.1]}

# Kita arahkan GridSearch untuk mencari model dengan AUC terbaik, bukan Akurasi
grid_auc = GridSearchCV(SVC(), param_grid_svc, cv=5, scoring='roc_auc')
grid_auc.fit(X_train, y_train)

print("Parameter terbaik berdasarkan AUC: {}".format(grid_auc.best_params_))
print("Skor AUC validasi silang terbaik: {:.3f}".format(grid_auc.best_score_))

# Evaluasi pada data test menggunakan AUC
print("Skor AUC pada data test: {:.3f}".format(
    roc_auc_score(y_test, grid_auc.decision_function(X_test))))